# Initial-margin CVaR MKABSDE

This notebook displays the retained quadrature campaign, both proxy BSDEs, and the common-path comparison of three conditional-CVaR estimators. Both payoffs use 256 full-law Gauss–Legendre nodes and Hermite degree two. The scheme solves for the MVA; future losses use the reconstructed total value.

## Complete reproduction commands

```bash
python -m initial_margin.run_campaign \
  --product all --steps 50 100 200 \
  --seeds 42 43 44 45 46 47 48 49 50 51 \
  --paths 200000 --degree 2 --device cuda

python -m initial_margin.compare_risk_estimators \
  --product all --steps 200 --seeds 42 43 44 \
  --paths 200000 --degree 2 --device cuda

python -m initial_margin.deterministic_reference \
  --product all --steps 50 100 200 --space-steps 6000 \
  --expectation-nodes 24 \
  --output initial_margin/results/deterministic_reference.json
```

In [1]:
import json
import sys
from pathlib import Path
from IPython.display import Markdown, display

root = Path.cwd().resolve()
while root != root.parent and not (root / 'pyproject.toml').is_file():
    root = root.parent
if not (root / 'pyproject.toml').is_file():
    raise FileNotFoundError('repository root not found')
sys.path.insert(0, str(root))
from initial_margin.evaluation import ARTICLE_REFERENCE_VALUES

result_root = root / 'initial_margin' / 'results'
products = ('call', 'butterfly')
reference_payload = json.loads((result_root / 'deterministic_reference.json').read_text())
direct_references = {(row['product'], row['time_steps']): row for row in reference_payload['rows']}
method_payload = json.loads((result_root / 'cvar_method_comparison.json').read_text())

## Relative margin valuation adjustment

Every number is $100\,\mathrm{MVA}_0/V_0^{bs}=100\,(V_0-V_0^{bs})/V_0^{bs}$, expressed in percent. The raw MVA and total price remain in the JSON artifacts; this is not the CVaR collateral.

In [2]:
for product in products:
    payload = json.loads((result_root / f'atm_{product}_experiment.json').read_text())
    base_price = payload['rows'][0]['metrics']['black_scholes_no_im']['price']
    article = ARTICLE_REFERENCE_VALUES[product]
    degree = payload['summary'][0]['hermite_degree']
    lines = [
        f'### ATM {product} — Hermite degree {degree}',
        '',
        '| $N$ | direct MKABSDE | deterministic reference | nonlinear proxy | linear proxy |',
        '|---:|---:|---:|---:|---:|',
    ]
    for row in payload['summary']:
        lines.append(
            f"| {row['time_steps']} | {row['direct_mkabsde_mva_percent_of_black_scholes_mean']:.6f} ± {row['direct_mkabsde_mva_percent_of_black_scholes_std']:.2e} | "
            f"{direct_references[(product, row['time_steps'])]['mva_percent_of_black_scholes']:.6f} | "
            f"{row['nonlinear_proxy_mva_percent_of_black_scholes_mean']:.6f} ± {row['nonlinear_proxy_mva_percent_of_black_scholes_std']:.2e} | "
            f"{row['linear_proxy_mva_percent_of_black_scholes_mean']:.6f} ± {row['linear_proxy_mva_percent_of_black_scholes_std']:.2e} |"
        )
    lines.append(
        f"| Reference value from Agarwal et al. | — | — | {100 * (article['nonlinear_proxy_price'] - base_price) / base_price:.6f} | {100 * (article['linear_proxy_price'] - base_price) / base_price:.6f} |"
    )
    display(Markdown('\n'.join(lines)))

### ATM call — Hermite degree 2

| $N$ | direct MKABSDE | deterministic reference | nonlinear proxy | linear proxy |
|---:|---:|---:|---:|---:|
| 50 | 0.861560 ± 1.31e-03 | 0.861569 | 1.010607 ± 1.32e-03 | 1.007173 ± 1.32e-03 |
| 100 | 0.859312 ± 9.07e-04 | 0.859493 | 1.007756 ± 9.32e-04 | 1.004309 ± 9.30e-04 |
| 200 | 0.858423 ± 1.10e-03 | 0.858241 | 1.006538 ± 1.14e-03 | 1.003083 ± 1.14e-03 |
| Reference value from Agarwal et al. | — | — | 1.002170 | 1.057365 |

### ATM butterfly — Hermite degree 2

| $N$ | direct MKABSDE | deterministic reference | nonlinear proxy | linear proxy |
|---:|---:|---:|---:|---:|
| 50 | 0.765830 ± 7.52e-04 | 0.766698 | 0.724722 ± 6.56e-04 | 0.723675 ± 6.56e-04 |
| 100 | 0.773317 ± 6.21e-04 | 0.773731 | 0.732213 ± 5.26e-04 | 0.731134 ± 5.27e-04 |
| 200 | 0.775954 ± 4.33e-04 | 0.776302 | 0.735188 ± 2.07e-04 | 0.734097 ± 2.07e-04 |
| Reference value from Agarwal et al. | — | — | 0.700663 | 0.700663 |

## Conditional-CVaR estimator audit

Only the CVaR engine changes below. The outer paths, Hermite regressions, and direct MKABSDE update are identical.

In [3]:
lines = [
    '| Product | CVaR estimator | $100\,\mathrm{MVA}_0/V_0^{bs}$ (%) | GPU time | price difference vs quadrature |',
    '|---|---|---:|---:|---:|',
]
for row in method_payload['summary']:
    lines.append(
        f"| {row['product']} | {row['method']} | {row['mva_percent_of_black_scholes_mean']:.6f} ± "
        f"{row['mva_percent_of_black_scholes_std']:.2e} | {row['gpu_seconds_mean']:.2f} s | "
        f"{row['price_difference_vs_quadrature_mean']:+.2e} |"
    )
display(Markdown('\n'.join(lines)))

| Product | CVaR estimator | $100\,\mathrm{MVA}_0/V_0^{bs}$ (%) | GPU time | price difference vs quadrature |
|---|---|---:|---:|---:|
| butterfly | quadrature | 0.775644 ± 2.61e-04 | 53.37 s | +0.00e+00 |
| butterfly | nested_mc | 0.777843 ± 7.81e-03 | 61.16 s | +6.80e-06 |
| butterfly | learning | 0.785767 ± 4.15e-03 | 54.95 s | +3.13e-05 |
| call | quadrature | 0.858587 ± 1.49e-03 | 37.69 s | +0.00e+00 |
| call | nested_mc | 0.859423 ± 1.21e-03 | 44.31 s | +1.82e-05 |
| call | learning | 0.860684 ± 2.61e-03 | 56.29 s | +4.56e-05 |

## Interpretation

The direct/proxy gap persists with all three conditional-risk estimators. Agreement of total prices within $10^{-4}$ does not make their much smaller MVAs identical. Quadrature evaluates the one-dimensional conditional law deterministically; measured GPU times are shown above. The deterministic state-space calculation is a separate fixed-grid check that also includes interpolation and quadrature errors, not an exact continuous-time solution.

## Larger margin periods

Only the margin horizon changes: Delta = 0.02, 0.10, 0.50 years. All runs use Hermite degree two, N = 200, 200,000 paths and three independent seeds. Entries are mean ± sample SD of 100 MVA / V_BS (%); the SD is in percentage points, not a numerical-error bound. These are stress tests of the small-Delta proxies, not usual financial margin periods.

    python -m initial_margin.margin_periods --device cuda --output-dir /tmp/absde-margin-table

Use a fresh output directory. The README in initial_margin lists the independent space/time/quadrature refinement commands. This notebook reads the retained JSON only.

In [4]:
horizons = json.loads((result_root / 'margin_periods' / 'summary.json').read_text())
lines = ['| Product | Delta (years) | Direct MKABSDE | Nonlinear proxy | Linear proxy |',
         '|---|---:|---:|---:|---:|']
for row in horizons:
    values = [f"{row['methods'][name]['mva_percent_mean']:.4f} ± {row['methods'][name]['mva_percent_std']:.4f}"
              for name in ('direct', 'nonlinear', 'linear')]
    lines.append(f"| {row['product']} | {row['margin_period']:.2f} | " + ' | '.join(values) + ' |')
display(Markdown('\n'.join(lines)))

| Product | Delta (years) | Direct MKABSDE | Nonlinear proxy | Linear proxy |
|---|---:|---:|---:|---:|
| call | 0.02 | 0.8586 ± 0.0015 | 1.0066 ± 0.0015 | 1.0032 ± 0.0015 |
| call | 0.10 | 1.5177 ± 0.0030 | 2.2000 ± 0.0033 | 2.1836 ± 0.0033 |
| call | 0.50 | 1.8903 ± 0.0042 | 4.2725 ± 0.0059 | 4.2114 ± 0.0059 |
| butterfly | 0.02 | 0.7756 ± 0.0003 | 0.7352 ± 0.0003 | 0.7341 ± 0.0003 |
| butterfly | 0.10 | 1.4921 ± 0.0008 | 1.5059 ± 0.0006 | 1.5016 ± 0.0006 |
| butterfly | 0.50 | 2.0144 ± 0.0010 | 2.4681 ± 0.0008 | 2.4592 ± 0.0008 |

In [5]:
checks = {}
for name in ('margin_period_reference', 'margin_period_double_q', 'margin_period_refined'):
    data = json.loads((result_root / name / 'summary.json').read_text())
    checks[name] = {(row['product'], row['margin_period']): row for row in data}
lines = ['| Product | Delta | Hermite vs same-grid | Doubling Q | Refined vs same-grid |',
         '|---|---:|---:|---:|---:|']
for row in horizons:
    key = (row['product'], row['margin_period'])
    direct = row['methods']['direct']['mva_mean']
    same, doubled, refined = [checks[name][key]['methods']['direct']['mva_mean'] for name in checks]
    lines.append(f"| {key[0]} | {key[1]:.2f} | {abs(direct-same):.2e} | {abs(doubled-same):.2e} | {abs(refined-same):.2e} |")
display(Markdown('Absolute differences in raw price units (not percentages):\n\n' + '\n'.join(lines)))

Absolute differences in raw price units (not percentages):

| Product | Delta | Hermite vs same-grid | Doubling Q | Refined vs same-grid |
|---|---:|---:|---:|---:|
| call | 0.02 | 7.52e-06 | 1.42e-07 | 1.68e-05 |
| call | 0.10 | 2.56e-05 | 2.42e-07 | 3.67e-05 |
| call | 0.50 | 3.70e-05 | 1.13e-07 | 5.08e-05 |
| butterfly | 0.02 | 2.04e-06 | 9.28e-08 | 3.26e-06 |
| butterfly | 0.10 | 2.34e-06 | 1.62e-07 | 1.69e-06 |
| butterfly | 0.50 | 1.39e-05 | 1.96e-08 | 3.33e-06 |

At Delta = 0.50, the direct MVA is about 56% below the nonlinear proxy for the call and 18.4% below for the butterfly. For the butterfly the gap changes sign: it was about 5.5% above at Delta = 0.02. Refinement preserves these large-horizon gaps, but neither the references nor the across-seed dispersion prove a uniform-in-Delta numerical error bound. The three-CVaR-engine comparison earlier in this notebook remains at Delta = 0.02.